**Name:** Srushti Supradeep Bansod  
**Section:** B3  
**Batch:** B2  
**Roll no.:** 29  

# SI LAB PROJECT
## Regression: Product Sales Prediction

**Aim:** To analyse e-commerce sales data using descriptive statistics, sampling, hypothesis testing and correlation, and then build regression models (Linear, Ridge, Lasso, Decision Tree, Random Forest) to predict the **Sales** of an order. The best model is saved and used in a Streamlit app.

**Dataset:** E-Commerce Sales Transactions Dataset (same dataset used in Practical 3b, via kagglehub).

**Note:** In this dataset `Sales` (total_amount) is calculated from price, discount and quantity, so these three columns will be the strongest predictors.

### Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

from scipy import stats
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Step 2: Load the Dataset

In [ ]:
import kagglehub
path = kagglehub.dataset_download("miadul/e-commerce-sales-transactions-dataset")
df = pd.read_csv(path + "/ecommerce_sales_34500.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
print(df.columns)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Rename target and profit columns (same as Practical 3b)
df.rename(columns = {"total_amount": "Sales", "profit_margin": "Profit"}, inplace = True)
df.columns

### Step 3: Descriptive Statistics of Sales

In [ ]:
sales = df["Sales"]

print("Mean              :", sales.mean())
print("Median            :", sales.median())
print("Mode              :", sales.mode()[0])
print("Variance          :", sales.var())
print("Standard Deviation:", sales.std())
print("Range             :", sales.max() - sales.min())

Q1 = sales.quantile(0.25)
Q3 = sales.quantile(0.75)
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", Q3 - Q1)
print("Skewness:", sales.skew())

Mean is much larger than the median and skewness is positive, so Sales is **right skewed** (a few very large orders).

### Step 4: Data Visualization

In [ ]:
plt.figure(figsize = (14, 4))

plt.subplot(1, 2, 1)
plt.hist(df["Sales"], bins = 50, color = "skyblue", edgecolor = "black")
plt.title("Distribution of Sales")
plt.xlabel("Sales")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
plt.boxplot(df["Sales"])
plt.title("Boxplot of Sales")
plt.ylabel("Sales")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize = (14, 4))

plt.subplot(1, 2, 1)
df.groupby("category")["Sales"].mean().sort_values().plot(kind = "bar", color = "lightgreen", edgecolor = "black")
plt.title("Average Sales by Category")
plt.ylabel("Mean Sales")

plt.subplot(1, 2, 2)
df.groupby("region")["Sales"].mean().sort_values().plot(kind = "bar", color = "coral", edgecolor = "black")
plt.title("Average Sales by Region")
plt.ylabel("Mean Sales")

plt.tight_layout()
plt.show()

### Step 5: Sampling, Central Limit Theorem and Confidence Interval

**Simple random sample vs population**

In [ ]:
sample = df["Sales"].sample(n = 500, random_state = 42)

print("Population Mean:", df["Sales"].mean())
print("Sample Mean    :", sample.mean())
print("Population Std :", df["Sales"].std())
print("Sample Std     :", sample.std())

**Central Limit Theorem:** Sales is not normal, but the distribution of sample means becomes approximately normal.

In [ ]:
population_mean = df["Sales"].mean()
population_std = df["Sales"].std()
sample_size = 50
number_of_samples = 1000
sample_means = []

for i in range(number_of_samples):
    s = df["Sales"].sample(n = sample_size, random_state = i)
    sample_means.append(s.mean())

print("Population Mean         :", population_mean)
print("Mean of Sample Means    :", np.mean(sample_means))
print("Theoretical Std. Error  :", population_std / np.sqrt(sample_size))
print("Observed Std. Error     :", np.std(sample_means, ddof = 1))

plt.figure(figsize = (8, 5))
plt.hist(sample_means, bins = 30, edgecolor = "black", color = "skyblue")
plt.axvline(population_mean, linestyle = "--", label = "Population Mean")
plt.xlabel("Sample Mean")
plt.ylabel("Frequency")
plt.title("Central Limit Theorem - Sampling Distribution of Sales")
plt.legend()
plt.show()

**95% Confidence Interval for mean Sales (t-distribution)**

In [ ]:
confidence_level = 0.95
mean = df["Sales"].mean()
sem = stats.sem(df["Sales"])

ci = stats.t.interval(confidence_level, len(df) - 1, loc = mean, scale = sem)
print("Standard Error of Mean:", sem)
print("95% Confidence Interval:", ci)

### Step 6: Hypothesis Testing

**1. One-sample t-test:** Is the mean Sales different from 150?  
H0: mean = 150, H1: mean ≠ 150

In [ ]:
t_stat, p_value = stats.ttest_1samp(df["Sales"], 150)
print("t-statistic:", t_stat)
print("p-value:", p_value)

alpha = 0.05
if p_value < alpha:
    print("Reject the Null Hypothesis")
else:
    print("Failed to reject the Null Hypothesis")

**2. Two-sample t-test:** Do male and female customers have different mean Sales?

In [ ]:
male = df[df["customer_gender"] == "Male"]["Sales"]
female = df[df["customer_gender"] == "Female"]["Sales"]

t_stat, p_value = stats.ttest_ind(male, female, equal_var = False)
print("Mean (Male)  :", male.mean())
print("Mean (Female):", female.mean())
print("t-statistic:", t_stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Reject the Null Hypothesis")
else:
    print("Failed to reject the Null Hypothesis")

**3. One-way ANOVA:** Is mean Sales different across regions?

In [ ]:
groups = [g["Sales"].values for _, g in df.groupby("region")]
f_stat, p_value = stats.f_oneway(*groups)
print("F-statistic:", f_stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Reject the Null Hypothesis")
else:
    print("Failed to reject the Null Hypothesis")

**4. Chi-Square test:** Is there an association between category and region?

In [ ]:
table = pd.crosstab(df["category"], df["region"])
chi2, p_value, dof, expected = stats.chi2_contingency(table)
print("Chi-square statistic:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", p_value)

if p_value < 0.05:
    print("Reject the Null Hypothesis")
else:
    print("Failed to reject the Null Hypothesis")

### Step 7: Correlation and Covariance

In [ ]:
num_cols = ["price", "discount", "quantity", "delivery_time_days", "customer_age", "Sales"]

print("Covariance with Sales:")
print(df[num_cols].cov()["Sales"])
print("\nCorrelation with Sales:")
print(df[num_cols].corr()["Sales"])

In [ ]:
plt.figure(figsize = (8, 6))
sns.heatmap(df[num_cols].corr(), annot = True, cmap = "coolwarm", fmt = ".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Pearson and Spearman with hypothesis testing
for col in ["price", "discount", "quantity"]:
    pr, pp = pearsonr(df[col], df["Sales"])
    sr, sp = spearmanr(df[col], df["Sales"])
    print("---", col, "vs Sales ---")
    print("Pearson  r =", round(pr, 4), " p-value =", pp)
    print("Spearman r =", round(sr, 4), " p-value =", sp)
    if pp < 0.05:
        print("Pearson: significant correlation\n")
    else:
        print("Pearson: not significant\n")

`quantity` and `price` show the strongest relationship with Sales, so they are expected to be the most important predictors.

### Step 8: Data Preprocessing

**Label Encoding** for categorical columns, then **train-test split (80/20)** and **Standard Scaling** (used for Ridge and Lasso).

In [ ]:
features = ["price", "discount", "quantity", "delivery_time_days", "customer_age",
            "category", "region", "payment_method", "customer_gender"]
categorical = ["category", "region", "payment_method", "customer_gender"]

data = df[features + ["Sales"]].copy()

encoders = {}
for col in categorical:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoders[col] = le

X = data[features]
y = data["Sales"]
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
print("Training Samples:", len(X_train))
print("Testing Samples :", len(X_test))

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Step 9: Build Regression Models

In [ ]:
# Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

# Ridge Regression (scaled data)
ridge_model = Ridge(alpha = 1.0)
ridge_model.fit(X_train_scaled, y_train)
ridge_pred = ridge_model.predict(X_test_scaled)

# Lasso Regression (scaled data)
lasso_model = Lasso(alpha = 0.1)
lasso_model.fit(X_train_scaled, y_train)
lasso_pred = lasso_model.predict(X_test_scaled)

# Decision Tree Regression
tree_model = DecisionTreeRegressor(max_depth = 8, random_state = 42)
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)

# Random Forest Regression
forest_model = RandomForestRegressor(n_estimators = 100, max_depth = 8, random_state = 42)
forest_model.fit(X_train, y_train)
forest_pred = forest_model.predict(X_test)

print("All models trained")

In [ ]:
print("Linear Regression Equation:")
for name, coef in zip(features, linear_model.coef_):
    print(f"  {name:20s}: {coef:.3f}")
print("  Intercept           :", round(linear_model.intercept_, 3))

### Step 10: Model Evaluation

In [ ]:
predictions = {
    "Linear Regression": linear_pred,
    "Ridge Regression": ridge_pred,
    "Lasso Regression": lasso_pred,
    "Decision Tree Regression": tree_pred,
    "Random Forest Regression": forest_pred
}

results = []
for model_name, prediction in predictions.items():
    mae = mean_absolute_error(y_test, prediction)
    mse = mean_squared_error(y_test, prediction)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, prediction)
    results.append([model_name, mae, mse, rmse, r2])

results_df = pd.DataFrame(results, columns = ["Model", "MAE", "MSE", "RMSE", "R2"])
print("Model Performance Comparison")
print(results_df.round(4).to_string(index = False))

In [ ]:
plt.figure(figsize = (14, 5))
plt.bar(results_df["Model"], results_df["RMSE"], color = "skyblue", edgecolor = "black")
plt.xlabel("Regression Model")
plt.ylabel("RMSE")
plt.title("RMSE Comparison of Regression Models")
plt.show()

plt.figure(figsize = (14, 5))
plt.bar(results_df["Model"], results_df["R2"], color = "lightgreen", edgecolor = "black")
plt.xlabel("Regression Model")
plt.ylabel("R2 Score")
plt.title("R2 Score Comparison of Regression Models")
plt.show()

### Step 11: Select the Best Model

In [ ]:
models = {
    "Linear Regression": linear_model,
    "Ridge Regression": ridge_model,
    "Lasso Regression": lasso_model,
    "Decision Tree Regression": tree_model,
    "Random Forest Regression": forest_model
}

best_model_name = results_df.loc[results_df["R2"].idxmax(), "Model"]
best_model = models[best_model_name]
print("Best Model:", best_model_name)
print("R2 Score  :", results_df["R2"].max())

In [ ]:
# Actual vs Predicted for the best model
if best_model_name in ["Ridge Regression", "Lasso Regression"]:
    best_pred = best_model.predict(X_test_scaled)
else:
    best_pred = best_model.predict(X_test)

plt.figure(figsize = (7, 6))
plt.scatter(y_test, best_pred, alpha = 0.4)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color = "red", linestyle = "--", label = "Perfect Prediction")
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Sales - " + best_model_name)
plt.legend()
plt.show()

comparison = pd.DataFrame({"Actual Sales": y_test.values, "Predicted Sales": best_pred}).head(10)
print(comparison.round(2).to_string(index = False))

### Step 12: Predict Sales for a New Order

In [ ]:
new_order = {
    "price": 150.0, "discount": 0.10, "quantity": 3, "delivery_time_days": 4, "customer_age": 30,
    "category": "Electronics", "region": "North", "payment_method": "UPI", "customer_gender": "Female"
}

new_df = pd.DataFrame([new_order])
for col in categorical:
    new_df[col] = encoders[col].transform(new_df[col])
new_df = new_df[features]

if best_model_name in ["Ridge Regression", "Lasso Regression"]:
    new_input = scaler.transform(new_df)
else:
    new_input = new_df

print("Predicted Sales:", round(best_model.predict(new_input)[0], 2))
print("Manual check (price x (1 - discount) x quantity):", 150.0 * 0.90 * 3)

### Step 13: Save the Model for Streamlit

In [ ]:
model_info = {
    "model": best_model,
    "name": best_model_name,
    "use_scaler": best_model_name in ["Ridge Regression", "Lasso Regression"],
    "features": features
}

joblib.dump(model_info, "best_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(encoders, "encoders.pkl")
results_df.round(4).to_csv("model_results.csv", index = False)
print("Saved: best_model.pkl, scaler.pkl, encoders.pkl, model_results.csv")

In [ ]:
# Download the files (run in Colab), keep them in the same folder as app.py
from google.colab import files
for f in ["best_model.pkl", "scaler.pkl", "encoders.pkl", "model_results.csv"]:
    files.download(f)

### Conclusion
- Sales is right skewed; the CLT shows that sample means of Sales are approximately normal.
- Price, quantity and discount are the main drivers of Sales (seen in the correlation analysis).
- Five regression models were compared using MAE, MSE, RMSE and R2, and the best model was selected using the highest R2 score.
- The best model is saved with joblib and used in a Streamlit app (`app.py`) to predict Sales for a new order.